# Task 5D — ONNX Export and CPU Benchmark
**Lab 06 | Original YOLOv8n-cls | 9 Nov 2026**

Run the cells in order. Original checkpoints and original Test images are read-only; outputs go to a new Drive directory. The notebook does not retrain a model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip -q install 'ultralytics>=8.3,<9' 'onnx>=1.15' 'onnxruntime>=1.17'

In [ ]:
from pathlib import Path
import urllib.request
base = Path('/content/drive/MyDrive/Lab06_results')
pt = base/'yolov8n/weights/best.pt'
test = base/'yolo_data/test'
manifest = base/'manifest.json'
for p in (pt, test, manifest):
    assert p.exists(), f'Missing expected original: {p}'
url = 'https://raw.githubusercontent.com/ID9999999999999/DL-Computer-vision/lab06-model-selection-2473986-20261025/LAB06_Model_Selection/scripts/task5d_export_onnx_benchmark.py'
script = Path('/content/task5d_export_onnx_benchmark.py')
urllib.request.urlretrieve(url,script)
print('Script ready:',script)
print('Original files verified, no training performed.')

In [ ]:
import subprocess,sys
out = base/'Task5D_2026-11-09'
cmd = [sys.executable, str(script), '--checkpoint',str(pt),'--test',str(test),'--manifest',str(manifest),'--out',str(out),'--threads','4']
print('Running export, paired full-Test evaluation and 20+100 CPU benchmarks...')
subprocess.run(cmd,check=True)
print('Results saved to',out)

In [ ]:
import json
from pathlib import Path
r=json.loads((out/'results_5d.json').read_text())
for k in ('original_pt','exported_onnx'):
    v=r[k]
    print(k, 'accuracy:',round(v['metrics']['accuracy'],6),'macro F1:',round(v['metrics']['macro_f1'],6), 'size MiB:',round(v['size_bytes']/2**20,3),'forward ms:',round(v['benchmark']['mean_ms'],3))
print('Prediction disagreements:',r['parity']['prediction_disagreements'])
print('Original 267/270 consistency:',r['baseline_archive_check']['expected_correct_267_of_270'])
print((out/'bonus_deployment_memo.md').read_text())